In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Embedding, Dense, TimeDistributed, Dropout, Bidirectional, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

In [2]:
from google.colab import files
uploaded = files.upload()

Saving ner_datasetreference.csv to ner_datasetreference.csv


In [3]:
df = pd.read_csv("ner_datasetreference.csv",encoding="latin1")
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In [6]:
df = df.drop(columns=['Tag'])

In [7]:
df

,Sentence #,Word,POS
0,Sentence: 1,Thousands,NNS
1,NaN,of,IN
2,NaN,demonstrators,NNS
3,NaN,have,VBP
4,NaN,marched,VBN
...,...,...,...
1048570,NaN,they,PRP
1048571,NaN,responded,VBD
1048572,NaN,to,TO
1048573,NaN,the,DT


In [8]:
df.size

3145725

In [9]:
df.shape

(1048575, 3)

In [10]:
class SentenceGetter(object):
    def __init__(self, data):
        self.data = data
        self.sentences = []
        self.grouped = self.data.groupby("Sentence #").apply(
            lambda s: [(w, p) for w, p in zip(s["Word"].values, s["POS"].values)]
        )
        self.sentences = [s for s in self.grouped]
getter = SentenceGetter(df)
sentences = getter.sentences

/tmp/ipython-input-3942873317.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.grouped = self.data.groupby("Sentence #").apply(


In [11]:
words = list(set(df["Word"].values))
words.append("PADword")

tags = list(set(df["POS"].values))

word2idx = {w: i + 1 for i, w in enumerate(words)}
tag2idx = {t: i for i, t in enumerate(tags)}

idx2word = {i: w for w, i in word2idx.items()}
idx2tag = {i: t for t, i in tag2idx.items()}


In [12]:
max_len = 50

X = [[word2idx[w[0]] for w in s] for s in sentences]
X = pad_sequences(maxlen=max_len, sequences=X, padding="post", value=0)

y = [[tag2idx[w[1]] for w in s] for s in sentences]
y = pad_sequences(maxlen=max_len, sequences=y, padding="post", value=tag2idx["NN"])

y = [to_categorical(i, num_classes=len(tags)) for i in y]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, np.array(y), test_size=0.2, random_state=42)

In [14]:
input = Input(shape=(max_len,))
model = Embedding(input_dim=len(words)+1, output_dim=50, input_length=max_len)(input)

model = Bidirectional(LSTM(units=100, return_sequences=True, recurrent_dropout=0.1))(model)
model = Dropout(0.5)(model)

out = TimeDistributed(Dense(len(tags), activation="softmax"))(model)

model = Model(input, out)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 50, 50)         │     1,759,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 200)        │       120,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 200)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 50, 42)         │         8,442 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,888,242 (7.20 MB)

 Trainable params: 1,888,242 (7.20 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=5,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 264s 235ms/step - accuracy: 0.9802 - loss: 0.1760 - val_accuracy: 0.9975 - val_loss: 0.0097
Epoch 2/5
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 249s 231ms/step - accuracy: 0.9981 - loss: 0.0078 - val_accuracy: 0.9988 - val_loss: 0.0044
Epoch 3/5
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 250s 231ms/step - accuracy: 0.9991 - loss: 0.0033 - val_accuracy: 0.9990 - val_loss: 0.0035
Epoch 4/5
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 259s 229ms/step - accuracy: 0.9994 - loss: 0.0022 - val_accuracy: 0.9991 - val_loss: 0.0033
Epoch 5/5
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 270s 237ms/step - accuracy: 0.9995 - loss: 0.0016 - val_accuracy: 0.9991 - val_loss: 0.0032


In [18]:
test_sentence = "The quick brown fox jumps over the lazy dog"

In [19]:
test_words = test_sentence.split()
test_seq = [word2idx.get(w, 0) for w in test_words]
test_seq = pad_sequences([test_seq], maxlen=max_len, padding="post")

In [20]:
prediction = model.predict(test_seq)
prediction = np.argmax(prediction, axis=-1)

for i, word in enumerate(test_words):
    print(word, ":", idx2tag[prediction[0][i]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
The : DT
quick : DT
brown : NN
fox : NN
jumps : NN
over : NN
the : NN
lazy : NN
dog : NN


In [21]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

300/300 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.9991 - loss: 0.0032
Test Accuracy: 0.9991475343704224
